# M06-P04 — LangGraph Enterprise Workflow

En M06-P03 vimos tool calling. Ahora convertimos esa secuencia en un **workflow empresarial explícito**.

```text
datos estructurados
+ guía operativa RAG
+ LLM
+ reglas de negocio
+ aprobación humana
```

El caso: Asteria Retail reporta fallos de autenticación en Managed VPN. El sistema debe preparar un briefing útil para un responsable de incidentes, pero detenerse antes de considerar aprobada cualquier comunicación externa sensible.


In [ ]:
import json
from pprint import pprint

from enterprise_genai.workflow import (
    start_incident_workflow,
    resume_incident_workflow,
    workflow_definition,
)

CUSTOMER = "Asteria Retail"
SERVICE = "Managed VPN"
QUESTION = (
    "Asteria Retail informa de fallos de autenticación en Managed VPN. "
    "Construye un briefing de incidente usando datos empresariales y guía operativa relevante. "
    "Prepara un borrador breve de actualización al cliente, pero no lo consideres aprobado "
    "sin revisión humana."
)

pprint(workflow_definition())


## 1. El grafo

LangGraph nos obliga a hacer explícitas piezas que en un bucle manual quedarían mezcladas:

```text
nodos
estado compartido
transiciones
routing condicional
checkpoint
interrupción
reanudación
```

Nuestro grafo es:

```text
START
  ↓
collect_enterprise_data
  ↓
retrieve_operational_guidance
  ↓
build_incident_brief
  ↓
¿P1 o servicio degradado?
  ├─ no → finalize → END
  └─ sí → human_review ⏸ → finalize → END
```


## 2. Iniciar el workflow

La primera ejecución debe avanzar sola hasta la puerta de aprobación.

En este laboratorio usamos `InMemorySaver`. El checkpoint queda asociado a un `thread_id`, que necesitaremos para continuar exactamente el mismo workflow.


In [ ]:
run = start_incident_workflow(
    customer_name=CUSTOMER,
    service_name=SERVICE,
    question=QUESTION,
)

print("Thread ID:", run["_thread_id"])
print("Interrumpido:", run["_interrupted"])
print("Estado:", run.get("workflow_status"))
print("Timeline:", run.get("timeline"))


## 3. El efecto útil: un paquete de decisión, no solo una respuesta

Antes de pedir aprobación, el workflow ya ha reunido:

- datos operativos actuales;
- incidentes;
- SLA contractual;
- documentación relevante;
- un briefing generado sobre evidencia;
- un borrador de comunicación.

Vamos a inspeccionarlo.


In [ ]:
print("=== DATOS EMPRESARIALES ===")
pprint(run["enterprise_data"])

print("\n=== FUENTES RECUPERADAS ===")
for source in run["retrieved_sources"]:
    print(
        source["source"],
        source["document_id"],
        source["title"],
        "score=", source["score"],
    )
    print(source["content"])
    print()

print("=== BRIEFING ===")
pprint(run["assessment"])


## 4. Routing: una regla de negocio, no una opinión del LLM

La aplicación decide si hace falta aprobación mediante una regla determinista.

En este caso se activa si hay un P1 o si el servicio no está operativo.

El modelo redacta; **la aplicación gobierna**.


In [ ]:
print("Requiere aprobación:", run["requires_human_approval"])
print("Motivo:", run["approval_reason"])
print("\nPayload de la interrupción:")
pprint(run["_interrupt_payload"])


## 5. Human-in-the-loop real

`interrupt()` ha detenido el grafo.

No estamos simulando una etiqueta `approved=True` dentro del mismo paso: LangGraph ha guardado el estado y está esperando una decisión externa.

Para continuar usamos:

```python
Command(resume=...)
```

El mismo `thread_id` identifica el checkpoint que debe retomarse.


### Decisión del revisor

En clase, revisa el briefing, las fuentes y el borrador antes de ejecutar la siguiente celda.

Puedes cambiar `APPROVE` a `False` y escribir un comentario de rechazo.


In [ ]:
APPROVE = True
REVIEW_COMMENT = "Revisado por Incident Manager. Los hechos y el lenguaje del borrador son adecuados."

completed = resume_incident_workflow(
    thread_id=run["_thread_id"],
    approved=APPROVE,
    comment=REVIEW_COMMENT,
)

print("Interrumpido:", completed["_interrupted"])
print("Estado final:", completed.get("workflow_status"))
print("Timeline:", completed.get("timeline"))


## 6. Resultado final

Si la persona aprueba, el workflow marca el briefing como aprobado para el siguiente paso.

Si rechaza, la comunicación externa queda retenida.

En ambos casos conservamos la evidencia y la trazabilidad.


In [ ]:
print(completed["final_output"])


## 7. Qué estado ha sobrevivido al pause/resume

El valor de LangGraph no es solo dibujar cajas.

El workflow ha conservado entre la pausa y la reanudación:

```text
cliente y servicio
datos de tools
fuentes recuperadas
briefing
regla de routing
borrador
decisión humana
timeline
```


In [ ]:
summary = {
    "customer": completed["customer_name"],
    "service": completed["service_name"],
    "status": completed["workflow_status"],
    "approval": completed.get("approval_decision"),
    "sources": [
        {
            "id": item["document_id"],
            "title": item["title"],
            "score": item["score"],
        }
        for item in completed["retrieved_sources"]
    ],
    "usage": completed.get("usage", {}),
}
pprint(summary)


## 8. Qué aporta LangGraph frente al bucle de P03

P03:

```text
LLM → tools → LLM
```

P04:

```text
workflow con estado
→ datos
→ retrieval
→ LLM
→ regla de routing
→ pausa
→ persona
→ reanudación
→ resultado
```

Esto es más cercano a un proceso empresarial real: no todo debe resolverse en una sola llamada al modelo y no toda decisión debe delegarse al modelo.


## 9. Probar la experiencia completa en Streamlit

Ejecuta:

```bash
streamlit run app/Home.py
```

En **Investigate**, selecciona:

```text
Workflow LangGraph (M06-P04)
```

Comprueba:

1. que aparecen los datos empresariales y las fuentes recuperadas;
2. que el workflow se detiene en **Aprobación humana requerida**;
3. que puedes aprobar o rechazar desde la GUI;
4. que, tras la decisión, el workflow continúa desde el mismo thread y finaliza;
5. que `Abrir la caja` muestra estado, timeline, routing, tokens y mecanismo de checkpoint.

El objetivo final no es el efecto visual por sí mismo. El “wow” empresarial es que, a partir de una sola petición, la aplicación reúne contexto de varias fuentes, prepara una decisión y **se detiene justo donde debe intervenir una persona**.
